# 🦷 Dental AI — U-Net Training Notebook

## 📖 Before You Start — Read This!

### What is Google Colab?
Google Colab is like a online computer that Google gives you for FREE.
It has a powerful GPU (graphics card) which makes training AI models very fast.
Your laptop does not have a GPU, so we train here instead.

### What is Mounting Google Drive?
Colab is an online computer — it cannot see files on your laptop directly.
So we first upload our dataset to **Google Drive**, then we **mount** it.
Mounting = connecting your Google Drive to Colab so Colab can read your files.
Think of it like plugging in a USB drive into a computer.

### What is a Cell?
This notebook is made of cells. Each cell has code inside it.
You run a cell by clicking the ▶ Play button on the left side of the cell.
Run cells ONE BY ONE from top to bottom. Wait for each to finish before running the next.
A cell is finished when you see a green checkmark ✅ or output appears below it.

---

## ⚠️ FIRST THING TO DO — Enable GPU
1. Click **Runtime** in the top menu
2. Click **Change runtime type**
3. Under Hardware Accelerator, select **T4 GPU**
4. Click **Save**

Do this BEFORE running any cell!

---

## 📋 What You Need To Do On Your Laptop Before Opening This Notebook:
1. Zip your `DentAI.v2i.coco-segmentation` folder
   - Right click the folder → Send to → Compressed (zip)
   - Make sure it is named exactly: `DentAI.v2i.coco-segmentation.zip`
2. Go to drive.google.com
3. Upload that zip file to My Drive (the main/root folder of your Drive)
4. Also make sure your code files are pushed to GitHub

---
## ✅ CELL 1 — Check GPU
This confirms Colab gave us a GPU for training.
You should see something like: `Tesla T4` in the output.
If it says `No GPU` then go back and enable T4 GPU in Runtime settings.

In [ ]:
import torch

print('GPU available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU name     :', torch.cuda.get_device_name(0))
    print('Everything is ready for training!')
else:
    print('NO GPU FOUND!')
    print('Go to Runtime > Change runtime type > Select T4 GPU > Save')
    print('Then run this cell again.')

---
## ✅ CELL 2 — Mount Google Drive

**What mounting means:**
Colab is an online computer. Your dataset zip is on Google Drive.
Mounting connects your Google Drive to this online computer
so that Colab can read your zip file.

**What will happen when you run this cell:**
A popup will appear asking you to sign in to your Google account.
Click the link, sign in, allow permission, copy the code and paste it back.
After this your Google Drive will be accessible at the path: `/content/drive/MyDrive/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted successfully!')
print('Your Drive files are now accessible at: /content/drive/MyDrive/')

---
## ✅ CELL 3 — Verify Dataset Zip Exists on Drive
This checks that your zip file is properly uploaded to Google Drive.
If it says ERROR, go back and upload the zip to your Google Drive root folder.

In [ ]:
import os

zip_path = '/content/drive/MyDrive/DentAI.v2i.coco-segmentation.zip'

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f'Dataset zip found on Drive!')
    print(f'File size: {size_mb:.1f} MB')
    print('Ready to proceed!')
else:
    print('ERROR: Zip file not found at:', zip_path)
    print('')
    print('Please:')
    print('1. Go to drive.google.com on your laptop')
    print('2. Upload DentAI.v2i.coco-segmentation.zip to My Drive (root folder)')
    print('3. Wait for upload to complete')
    print('4. Run this cell again')

---
## ✅ CELL 4 — Clone Your GitHub Repository

**What cloning means:**
Cloning downloads all your code files from GitHub into this Colab computer.
This is how Colab gets your `train.py`, `model.py`, `dataset.py` etc.

**Change the values below to your actual GitHub username and repo name.**

Your repo name is `Dental_AI` based on your folder name.
Your username is whatever you used to sign up on GitHub.

In [ ]:
# ⚠️ CHANGE THESE TWO VALUES
GITHUB_USERNAME = 'yourusername'    # e.g. 'yeshfa-hussain'
GITHUB_REPO     = 'Dental_AI'      # e.g. 'Dental_AI'

# Clone the repo
%cd /content
!git clone https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git

# Move into the project folder
%cd /content/{GITHUB_REPO}

# Show what files were downloaded
print('\nFiles in your project:')
!ls

---
## ✅ CELL 5 — Unzip Dataset into Project Folder

This copies your dataset from Google Drive into the Colab computer
and extracts (unzips) it so the training code can read the images.

This may take 1-3 minutes depending on dataset size. Please wait.

In [ ]:
import os

REPO_PATH   = f'/content/{GITHUB_REPO}'
zip_path    = '/content/drive/MyDrive/DentAI.v2i.coco-segmentation.zip'

print('Unzipping dataset... please wait...')
!unzip -q "{zip_path}" -d "{REPO_PATH}/"
print('Unzip complete!')

# Verify folders exist
print('\nVerifying folder structure:')
for split in ['train', 'test', 'valid']:
    path  = os.path.join(REPO_PATH, 'DentAI.v2i.coco-segmentation', split)
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        print(f'  {split}: {len(files)} images found ✅')
    else:
        print(f'  {split}: FOLDER NOT FOUND ❌ — check your zip structure')

---
## ✅ CELL 6 — Install Dependencies

Colab already has PyTorch installed.
We only need to install `pycocotools` which is used for reading COCO annotations.

In [ ]:
!pip install pycocotools -q
print('pycocotools installed!')

# Verify all imports work
import torch
import numpy as np
from PIL import Image
from pycocotools import mask as coco_mask
print('All imports working!')
print('PyTorch version:', torch.__version__)

---
## ✅ CELL 7 — Convert COCO Annotations to PNG Masks

This runs your `coco_to_masks.py` script.
It reads the annotation JSON files and creates PNG mask images.
This is the same step you already tested on your laptop.

It will create a `masks/` folder with train, test, valid subfolders.

In [ ]:
%cd /content/{GITHUB_REPO}
!python coco_to_masks.py

In [ ]:
# Verify masks were created
import os
print('Verifying masks:')
for split in ['train', 'test', 'valid']:
    mask_dir = os.path.join(REPO_PATH, 'masks', split)
    if os.path.exists(mask_dir):
        count = len([f for f in os.listdir(mask_dir) if f.endswith('.png')])
        print(f'  masks/{split}: {count} masks ✅')
    else:
        print(f'  masks/{split}: NOT FOUND ❌')

---
## ✅ CELL 8 — Test Dataset Loading

Before training, let's make sure images and masks load correctly.
You should see image shape `(1, 512, 512)` and mask shape `(512, 512)`.

In [ ]:
import sys
sys.path.insert(0, REPO_PATH)

from dataset import DentalDataset

train_dataset = DentalDataset(
    image_dir = os.path.join(REPO_PATH, 'DentAI.v2i.coco-segmentation', 'train'),
    mask_dir  = os.path.join(REPO_PATH, 'masks', 'train'),
)

img, mask = train_dataset[0]
print(f'Image shape  : {img.shape}')    # should be (1, 512, 512)
print(f'Mask shape   : {mask.shape}')   # should be (512, 512)
print(f'Mask classes : {mask.unique()}') # should have values from 0,1,2,3
print('Dataset loading works correctly!')

---
## ✅ CELL 9 — START TRAINING

**This is the main step!**

What will happen:
- Training will run for 50 epochs
- Each epoch = the model sees your entire dataset once
- After each epoch you will see: Train Loss, Val Loss, IoU, Dice scores
- Loss going DOWN = model is learning ✅
- Best model is automatically saved to `checkpoints/best_model.pth`

**How long will it take?**
Approximately 30-60 minutes on T4 GPU. Do not close the browser tab!

**What is Loss?** A number showing how wrong the model is. Lower = better.
**What is IoU?** Measures how well predicted regions overlap with real ones. Higher = better.
**What is Dice?** Similar to IoU, another accuracy measure. Higher = better.

In [ ]:
%cd /content/{GITHUB_REPO}
!python train.py

---
## ✅ CELL 10 — View Training Curves

After training, this shows you a graph of:
- Loss over epochs (should go down)
- IoU over epochs (should go up)
- Dice over epochs (should go up)

In [ ]:
from IPython.display import Image as IPImage
import os

curves_path = os.path.join(REPO_PATH, 'checkpoints', 'training_curves.png')

if os.path.exists(curves_path):
    display(IPImage(curves_path, width=900))
else:
    print('Training curves not found. Did training complete?')

---
## ✅ CELL 11 — Save Model to Google Drive

Colab computers are temporary — they get deleted when you close the tab.
So we MUST save the trained model to Google Drive before closing.

After this cell runs, go to drive.google.com and you will see `best_model.pth` there.
Download it to your laptop and place it in `Dental_AI/checkpoints/` folder.

In [ ]:
import shutil
import os

DRIVE_SAVE_PATH = '/content/drive/MyDrive'

# Save best model
src_model = os.path.join(REPO_PATH, 'checkpoints', 'best_model.pth')
dst_model = os.path.join(DRIVE_SAVE_PATH, 'best_model.pth')

if os.path.exists(src_model):
    shutil.copy(src_model, dst_model)
    size_mb = os.path.getsize(dst_model) / (1024 * 1024)
    print(f'best_model.pth saved to Google Drive! ({size_mb:.1f} MB)')
else:
    print('ERROR: best_model.pth not found. Did training complete successfully?')

# Save training curves image
src_curves = os.path.join(REPO_PATH, 'checkpoints', 'training_curves.png')
dst_curves = os.path.join(DRIVE_SAVE_PATH, 'training_curves.png')

if os.path.exists(src_curves):
    shutil.copy(src_curves, dst_curves)
    print('training_curves.png saved to Google Drive!')

print('\nNow go to drive.google.com and download best_model.pth to your laptop!')

---
## ✅ CELL 12 — Quick Prediction Test in Colab (Optional)

Test the trained model on one X-ray image right here before going back to laptop.

In [ ]:
import os

test_img_dir = os.path.join(REPO_PATH, 'DentAI.v2i.coco-segmentation', 'test')
test_images  = [f for f in os.listdir(test_img_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

if test_images:
    sample = os.path.join(test_img_dir, test_images[0])
    print(f'Testing on: {test_images[0]}')
    !python predict.py --image "{sample}"
else:
    print('No test images found.')

In [ ]:
# Show the colorized prediction result
import glob
from IPython.display import Image as IPImage

pred_files = glob.glob(os.path.join(REPO_PATH, 'predictions', '*_comparison.png'))

if pred_files:
    print('Prediction result:')
    print('Red=Caries | Yellow=Infection | Green=Restoration | Black=Background')
    display(IPImage(pred_files[0], width=900))
else:
    print('No prediction images found.')

---
## 🎯 After Training — What To Do On Your Laptop

1. Go to **drive.google.com**
2. Download **best_model.pth**
3. In your `Dental_AI` project folder, create a folder called `checkpoints`
4. Place `best_model.pth` inside it:
```
Dental_AI/
└── checkpoints/
    └── best_model.pth
```
5. Open terminal in VSCode and run:
```bash
python predict.py --image path\to\any\xray.jpg
```
6. Check the `predictions/` folder for your colorized output!

---
**Color Legend:**
- 🔴 Red = Caries
- 🟡 Yellow = Infection
- 🟢 Green = Restoration
- ⬛ Black = Background